<a href="https://colab.research.google.com/github/Ilham-sy/psa-nlp-project/blob/main/NLP_GRP_PRJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PSA Translation Project

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd


# Define the base path for files in Google Drive
drive_path = "/content/drive/MyDrive/"

multilanguage = pd.read_csv(f"{drive_path}Multilanguage_PSA.csv")
PSA = pd.read_csv(f"{drive_path}PSA_KE_Final.csv")

In [ ]:
# Load the dataset
# This cell is redundant as data is loaded in StYVbsX22qZn
# multilanguage = pd.read_csv("Multilanguage_PSA.csv")
# PSA = pd.read_csv("PSA_KE_Final.csv")

In [ ]:
# Check the dimensions
print("Multilanguage:", multilanguage.shape)
print("PSA:", PSA.shape)

Multilanguage: (6648, 9)
PSA: (2903, 8)


In [ ]:
# Check the columns
print("Multilanguage columns:")
print(multilanguage.columns.tolist())

print("\nPSA columns:")
print(PSA.columns.tolist())

Multilanguage columns:
['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Source', 'Date', 'Metadata', 'PSA ID']

PSA columns:
['PSA_Id', 'Domain', 'Class', 'English', 'Kiswahili', 'Ekegusii', 'Dholuo', 'Somali']


In [ ]:
# Drop unnecessary columns for multilanguage
multilanguage = multilanguage.drop(columns=["PSA ID", "Source", "Date", "Metadata"])

In [ ]:
# Drop unnecessary columns for PSA
PSA = PSA.drop(columns=["Class", "Ekegusii", "Somali"])

In [ ]:
# Rename columns
PSA = PSA.rename(columns={"PSA_Id": "PSA_ID"})

In [ ]:
# Combine the datasets
combined = pd.concat([multilanguage, PSA], ignore_index=True)

In [ ]:
# check the combined dataset
print(combined.shape)

combined.head()

(9551, 5)


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN


In [ ]:
# Assign the combined DataFrame to 'df' for the cleaning process
df = combined

In [ ]:
combined.to_csv(
    f"{drive_path}Combined_PSA_Raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Structural Cleaning

In [ ]:
from google.colab import drive
import pandas as pd
import re

In [ ]:
# Merge duplicate ID columns (if both exist)
if 'PSA ID' in df.columns and 'PSA_ID' in df.columns:
    df['PSA_ID'] = df['PSA_ID'].fillna(df['PSA ID'])
    df = df.drop(columns=['PSA ID'])

TEXT_COLS = ['English', 'Kiswahili', 'Dholuo']

def clean_text(val):
    if pd.isna(val):
        return val

    text = str(val)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove HTML entities
    text = re.sub(r'&nbsp;|&amp;|&quot;|&#\d+;', ' ', text)

    # Clean special character encoding
    text = re.sub(r'_x0092_', "'", text)   # right single quote
    text = re.sub(r'_x0093_', '"', text)   # left double quote
    text = re.sub(r'_x0094_', '"', text)   # right double quote
    text = re.sub(r'_x0096_', '-', text)   # en dash
    text = re.sub(r'_x0097_', '-', text)   # em dash
    text = re.sub(r'_x00[0-9A-Fa-f]{2}_', ' ', text)  # catch-all

    # Remove line breaks and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # Convert curly quotes to straight quotes
    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")

    # Remove extra spaces around brackets
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Collapse multiple spaces
    text = re.sub(r'\s{2,}', ' ', text)

    # Trim whitespace
    text = text.strip()

    return text

# Apply cleaning
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# Remove rows with empty English text
if 'English' in df.columns:
    df = df[df['English'].notna() & (df['English'].str.strip() != '')]

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()  # exact full-row duplicates (safe to keep)

# Remove content-duplicate rows (same PSA under a different PSA_ID)
dedup_cols = ['English', 'Kiswahili'] # Removed 'Source' and 'Date'
df = df.drop_duplicates(subset=dedup_cols, keep='first')

print(f"Duplicate rows removed: {before - len(df)}")

# Save cleaned dataset back to Google Drive
output_path = '/content/drive/MyDrive/PSA_Clean_v1.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Final rows: {len(df)}")
print(f"Saved as: {output_path}")

# Preview cleaned data
df.head()

Duplicate rows removed: 7
Final rows: 9540
Saved as: /content/drive/MyDrive/PSA_Clean_v1.csv


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN
